In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _read_trace_csv_1d(csv_path):
    arr = pd.read_csv(csv_path).to_numpy(dtype=float).ravel()
    arr = np.asarray(arr, dtype=float)
    return arr[np.isfinite(arr)]


def _find_existing_file(base_path, candidates):
    for name in candidates:
        p = base_path / name
        if p.is_file():
            return p
    return None


def _gaussian_smooth_1d(x, sigma_samples):
    x = np.asarray(x, dtype=float).ravel()
    sigma_samples = float(sigma_samples)
    if x.size == 0 or (not np.isfinite(sigma_samples)) or sigma_samples <= 0:
        return x.copy()
    radius = max(1, int(np.ceil(3.0 * sigma_samples)))
    k = np.arange(-radius, radius + 1, dtype=float)
    w = np.exp(-0.5 * (k / sigma_samples) ** 2)
    w /= np.sum(w)
    return np.convolve(x, w, mode='same')


def _safe_axis_limits(y, pad_frac=0.08, fallback=(0.0, 1.0)):
    y = np.asarray(y, dtype=float).ravel()
    y = y[np.isfinite(y)]
    if y.size == 0:
        return [float(fallback[0]), float(fallback[1])]

    lo = float(np.min(y))
    hi = float(np.max(y))
    span = hi - lo

    if not np.isfinite(lo) or not np.isfinite(hi):
        return [float(fallback[0]), float(fallback[1])]

    if span <= 0:
        pad = max(1e-3, 0.05 * abs(hi) + 1e-3)
    else:
        pad = max(1e-9, float(pad_frac) * span)

    return [lo - pad, hi + pad]


def plot_cal_vol_first60_for_figure(
    cell_path,
    cal_sr,
    start_s=0.0,
    vol_sr=500.0,
    window_s=60.0,
    save_stem='cal_vol_first60_forFigure',
    add_overlay_panel=True,
    cal_smooth_sigma_s=0.05,
):
    """
    Paper-style trace figure:
    - row 1: voltage only (magenta)
    - row 2: calcium only (green)
    - row 3 (optional): overlay voltage+calcium with separate y-axes
    Saves HTML and SVG in cell folder.
    """
    cell_path = Path(cell_path)
    if not cell_path.is_dir():
        raise FileNotFoundError(f'Folder not found: {cell_path}')

    cal_file = _find_existing_file(cell_path, ['calDF.csv', 'calTraceDF.csv'])
    vol_file = _find_existing_file(cell_path, ['volDF.csv', 'volTraceDF.csv'])

    if cal_file is None:
        raise FileNotFoundError(f'No calcium file found in {cell_path} (expected calDF.csv or calTraceDF.csv)')
    if vol_file is None:
        raise FileNotFoundError(f'No voltage file found in {cell_path} (expected volDF.csv or volTraceDF.csv)')

    cal = _read_trace_csv_1d(cal_file)
    vol = _read_trace_csv_1d(vol_file)

    cal_sr = float(cal_sr)
    start_s = float(start_s)
    vol_sr = float(vol_sr)
    window_s = float(window_s)
    cal_smooth_sigma_s = float(cal_smooth_sigma_s)
    end_s = start_s + window_s

    i0 = max(0, int(round(start_s * vol_sr)))
    i1 = min(len(vol), int(round(end_s * vol_sr)))
    if i1 <= i0:
        raise ValueError('Selected window is outside voltage trace range.')

    ci0 = max(0, int(round(start_s * cal_sr)))
    ci1 = min(len(cal), int(round(end_s * cal_sr)))
    if ci1 <= ci0:
        raise ValueError('Selected window is outside calcium trace range.')

    cal_60 = cal[ci0:ci1]
    cal_sigma_samples = max(0.0, cal_smooth_sigma_s * cal_sr)
    cal_60_smooth = _gaussian_smooth_1d(cal_60, cal_sigma_samples)
    vol_60 = vol[i0:i1]

    t_cal = np.arange(ci0, ci1, dtype=float) / cal_sr
    t_vol = np.arange(i0, i1, dtype=float) / vol_sr

    # Robust limits (finite-only) and consistent voltage scaling across panels
    v_rng = _safe_axis_limits(vol_60, pad_frac=0.06)
    # Add explicit top headroom so spikes do not look visually capped at the panel ceiling.
    v_span = max(1e-9, float(v_rng[1] - v_rng[0]))
    v_rng[1] = float(v_rng[1] + 0.12 * v_span)
    c_rng = _safe_axis_limits(cal_60_smooth, pad_frac=0.10)

    # Keep overlay voltage on the exact same scale as row 1 to prevent visual mismatch/capping
    v_ov_rng = list(v_rng)
    c_ov_rng = _safe_axis_limits(cal_60_smooth, pad_frac=0.08)

    if add_overlay_panel:
        fig = make_subplots(
            rows=3,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            row_heights=[0.31, 0.31, 0.38],
            specs=[[{}], [{}], [{'secondary_y': True}]],
        )
    else:
        fig = make_subplots(
            rows=2,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.06,
            row_heights=[0.5, 0.5],
            specs=[[{}], [{}]],
        )

    # Row 1: voltage
    fig.add_trace(
        go.Scatter(
            x=t_vol,
            y=vol_60,
            mode='lines',
            line=dict(color='#d62728', width=1.1),
            name='Voltage',
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    # Row 2: calcium
    fig.add_trace(
        go.Scatter(
            x=t_cal,
            y=cal_60_smooth,
            mode='lines',
            line=dict(color='mediumseagreen', width=1.4),
            name='Calcium',
            showlegend=False,
        ),
        row=2,
        col=1,
    )

    # Row 3: overlay with dual y-axis
    if add_overlay_panel:
        fig.add_trace(
            go.Scatter(
                x=t_vol,
                y=vol_60,
                mode='lines',
                line=dict(color='#d62728', width=1.1),
                name='Voltage (overlay)',
                showlegend=False,
            ),
            row=3,
            col=1,
            secondary_y=False,
        )
        fig.add_trace(
            go.Scatter(
                x=t_cal,
                y=cal_60_smooth,
                mode='lines',
                line=dict(color='mediumseagreen', width=1.4),
                name='Calcium (overlay)',
                showlegend=False,
            ),
            row=3,
            col=1,
            secondary_y=True,
        )

    common_axis_style = dict(
        showgrid=False,
        showline=True,
        linewidth=2.6,
        linecolor='black',
        mirror=False,
        ticks='outside',
        tickwidth=2.4,
        ticklen=7,
        tickfont=dict(size=14, family='Helvetica', color='black'),
    )
    fig.update_xaxes(**common_axis_style)
    fig.update_yaxes(**common_axis_style)

    if add_overlay_panel:
        fig.update_xaxes(title_text='<b>Time (s)</b>', title_font=dict(size=18, family='Helvetica', color='black'), range=[start_s, end_s], row=3, col=1)
    else:
        fig.update_xaxes(title_text='<b>Time (s)</b>', title_font=dict(size=18, family='Helvetica', color='black'), range=[start_s, end_s], row=2, col=1)

    fig.update_yaxes(title_text='<b>Voltage</b>', title_font=dict(size=18, family='Helvetica', color='black'), range=v_rng, row=1, col=1)
    fig.update_yaxes(title_text='<b>Calcium dF/F</b>', title_font=dict(size=18, family='Helvetica', color='black'), range=c_rng, row=2, col=1)

    if add_overlay_panel:
        fig.update_yaxes(title_text='<b>Voltage (overlay)</b>', title_font=dict(size=16, family='Helvetica', color='black'), range=v_ov_rng, row=3, col=1, secondary_y=False)
        fig.update_yaxes(title_text='<b>Calcium dF/F (overlay)</b>', title_font=dict(size=16, family='Helvetica', color='black'), range=c_ov_rng, row=3, col=1, secondary_y=True)

    fig.update_layout(
        template='simple_white',
        width=1500,
        height=760 if add_overlay_panel else 520,
        font=dict(family='Helvetica', size=14, color='black'),
        showlegend=False,
        margin=dict(l=95, r=75, t=25, b=75),
    )

    # Re-assert voltage ranges at the end to guarantee row1 and overlay left axis match exactly.
    fig.update_yaxes(range=v_rng, row=1, col=1)
    if add_overlay_panel:
        fig.update_yaxes(range=v_rng, row=3, col=1, secondary_y=False)

    html_path = cell_path / f'{save_stem}.html'
    svg_path = cell_path / f'{save_stem}.svg'

    fig.write_html(str(html_path))
    try:
        fig.write_image(str(svg_path))
    except Exception as e:
        print(f'[WARN] SVG export failed: {e}')

    print('Saved HTML:', html_path)
    print('Saved SVG :', svg_path)
    return fig, html_path, svg_path


# ===== Example usage =====
CELL_PATH = r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\srugc17\Rb\14-07-2025\fov8\cell1'  # <-- change
CAL_SR = 30  # <-- change
START_S = 0  # <-- choose start time (s)

fig, html_path, svg_path = plot_cal_vol_first60_for_figure(
    cell_path=CELL_PATH,
    cal_sr=CAL_SR,
    start_s=START_S,
    vol_sr=500.0,
    window_s=30.0,
    cal_smooth_sigma_s=0.03,
    save_stem='cal_vol_first60_forFigure',
    add_overlay_panel=True,
)


Saved HTML: Z:\Adam-Lab-Shared\Data\Michal_Rubin\srugc17\Rb\14-07-2025\fov8\cell1\cal_vol_first60_forFigure.html
Saved SVG : Z:\Adam-Lab-Shared\Data\Michal_Rubin\srugc17\Rb\14-07-2025\fov8\cell1\cal_vol_first60_forFigure.svg


In [7]:
import os
import re
import glob
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _read_trace_csv_1d(csv_path):
    arr = pd.read_csv(csv_path).to_numpy(dtype=float).ravel()
    arr = np.asarray(arr, dtype=float)
    return arr[np.isfinite(arr)]


def _find_existing_file(base_path, candidates):
    for name in candidates:
        p = base_path / name
        if p.is_file():
            return p
    return None


def _find_spike_pkl(cell_path, preferred_name='spike_detection_redefine_new.pkl'):
    cell_path = Path(cell_path)

    # 1) Use requested exact file first (if present)
    if preferred_name:
        p = cell_path / preferred_name
        if p.is_file():
            return p

    # 2) Then fallback to common names
    pref = [
        cell_path / 'final_spikes.pkl',
        cell_path / 'spike_detection_refined_new.pkl',
        cell_path / 'final_correct_spike_detection.pkl',
    ]
    for p in pref:
        if p.is_file():
            return p

    cands = []
    if preferred_name:
        base = preferred_name.replace('.pkl', '')
        cands.extend(glob.glob(str(cell_path / f'{base}*.pkl')))
    cands.extend(glob.glob(str(cell_path / 'final_spikes*.pkl')))
    cands.extend(glob.glob(str(cell_path / 'spike_detection_refined_new*.pkl')))
    cands.extend(glob.glob(str(cell_path / 'final_correct_spike_detection*.pkl')))
    if len(cands) == 0:
        return None

    def _nat_key(path):
        name = os.path.basename(path).lower()
        if preferred_name and preferred_name.lower() in name:
            pref_rank = 0
        elif name.startswith('final_spikes'):
            pref_rank = 1
        elif name.startswith('spike_detection_refined_new'):
            pref_rank = 2
        elif name.startswith('final_correct_spike_detection'):
            pref_rank = 3
        else:
            pref_rank = 4

        m = re.search(r'(\d+)(?=\.pkl$)', name)
        num = int(m.group(1)) if m else -1
        return (pref_rank, num, name)

    cands = sorted(set(cands), key=_nat_key)
    return Path(cands[0])


def _unique_sorted_in_bounds(x, n):
    if x is None:
        return np.array([], dtype=int)
    try:
        a = np.asarray(x, dtype=int).ravel()
    except Exception:
        return np.array([], dtype=int)
    if a.size == 0:
        return np.array([], dtype=int)
    a = a[(a >= 0) & (a < int(n))]
    if a.size == 0:
        return np.array([], dtype=int)
    return np.unique(a)


def _enforce_refractory(idx, min_frames=2):
    idx = np.asarray(idx, dtype=int)
    if idx.size == 0:
        return idx
    keep = [int(idx[0])]
    for v in idx[1:]:
        if int(v) - keep[-1] >= int(min_frames):
            keep.append(int(v))
    return np.asarray(keep, dtype=int)


def _extract_spikes_from_pkl_payload(payload, n_vol):
    # final_spikes.pkl can be a bare index vector/list
    if not isinstance(payload, dict):
        return _unique_sorted_in_bounds(payload, n_vol)

    # Preferred consolidated keys
    for k in (
        'spike_indices',
        'vm_all_spikes',
        'all_spikes',
        'spikes',
        'spike_idx',
        'spikeID',
        'refined_SS',
    ):
        if k in payload:
            arr = _unique_sorted_in_bounds(payload.get(k), n_vol)
            if arr.size > 0:
                return arr

    # Fallback: combine simple+complex labels if present
    simple = _unique_sorted_in_bounds(payload.get('vm_simple_spike', []), n_vol)
    complex_ = _unique_sorted_in_bounds(payload.get('vm_complex_spike', []), n_vol)
    if simple.size > 0 or complex_.size > 0:
        return np.unique(np.r_[simple, complex_]).astype(int)

    return np.array([], dtype=int)


def _fallback_detect_spikes_from_voltage(vol, min_dist_frames=2):
    vol = np.asarray(vol, dtype=float).ravel()
    if vol.size < 3:
        return np.array([], dtype=int)
    dv = np.abs(np.diff(vol))
    thr = float(np.nanpercentile(dv[np.isfinite(dv)], 99.5)) if np.any(np.isfinite(dv)) else np.nan
    if not np.isfinite(thr) or thr <= 0:
        return np.array([], dtype=int)
    idx = np.where(dv > thr)[0] + 1
    idx = _unique_sorted_in_bounds(idx, len(vol))
    idx = _enforce_refractory(idx, min_frames=min_dist_frames)
    return idx


def _estimate_segment_offset_in_full_trace(full_vol, seg_vol):
    full = np.asarray(full_vol, dtype=float).ravel()
    seg = np.asarray(seg_vol, dtype=float).ravel()
    if full.size < 10 or seg.size < 10 or seg.size >= full.size:
        return 0

    # Downsample for fast coarse alignment.
    step = max(1, int(round(seg.size / 8000.0)))
    f = full[::step]
    g = seg[::step]

    mf = np.isfinite(f)
    mg = np.isfinite(g)
    if np.sum(mf) < 10 or np.sum(mg) < 10:
        return 0

    f = np.where(np.isfinite(f), f, np.nanmedian(f[mf]))
    g = np.where(np.isfinite(g), g, np.nanmedian(g[mg]))

    fs = np.nanstd(f)
    gs = np.nanstd(g)
    if (not np.isfinite(fs)) or fs <= 1e-12 or (not np.isfinite(gs)) or gs <= 1e-12:
        return 0

    fz = (f - np.nanmean(f)) / fs
    gz = (g - np.nanmean(g)) / gs

    if fz.size < gz.size:
        return 0

    corr = np.correlate(fz, gz, mode='valid')
    if corr.size == 0 or not np.any(np.isfinite(corr)):
        return 0

    coarse = int(np.nanargmax(corr)) * step

    # Small local refinement on original traces.
    n = int(seg.size)
    lo = max(0, coarse - 3 * step)
    hi = min(int(full.size - n), coarse + 3 * step)
    if hi < lo:
        return max(0, min(int(full.size - n), coarse))

    best_off = coarse
    best_err = np.inf
    for off in range(lo, hi + 1):
        a = full[off:off + n]
        m = np.isfinite(a) & np.isfinite(seg)
        if np.sum(m) < 20:
            continue
        err = float(np.nanmean((a[m] - seg[m]) ** 2))
        if np.isfinite(err) and err < best_err:
            best_err = err
            best_off = off

    best_off = int(max(0, min(int(full.size - n), best_off)))
    return best_off


def _align_spikes_to_full_trace_if_needed(spike_idx, payload, full_vol, pkl_path=None, vol_sr=500.0):
    sp = _unique_sorted_in_bounds(spike_idx, len(full_vol))
    if not isinstance(payload, dict):
        return sp, 0

    seg_vol = np.asarray(payload.get('trace_vol', []), dtype=float).ravel()
    if seg_vol.size <= 0 or seg_vol.size >= len(full_vol):
        return sp, 0

    # If spikes already clearly outside segment-local range, assume already full-trace aligned.
    if sp.size > 0 and np.max(sp) >= seg_vol.size:
        return sp, 0

    off = _estimate_segment_offset_in_full_trace(full_vol, seg_vol)
    if off <= 0:
        return sp, 0

    sp_aligned = _unique_sorted_in_bounds(sp + int(off), len(full_vol))
    try:
        name = Path(pkl_path).name if pkl_path is not None else 'pkl'
        print(f'[ALIGN] {name}: shifted spikes by +{off} frames ({off/float(vol_sr):.3f}s) to full-trace timeline')
    except Exception:
        pass
    return sp_aligned, int(off)


def _choose_best_pkl_for_window(cell_path, preferred_name, full_vol, i0, i1, vol_sr=500.0):
    cell_path = Path(cell_path)

    cand = []
    if preferred_name:
        exact = cell_path / str(preferred_name)
        if exact.is_file():
            cand.append(exact)

    # For motor cells with segmented files, evaluate all refined variants by overlap with window.
    cand.extend([Path(x) for x in glob.glob(str(cell_path / 'spike_detection_refined_new*.pkl'))])
    cand.extend([Path(x) for x in glob.glob(str(cell_path / 'final_correct_spike_detection*.pkl'))])
    cand.extend([Path(x) for x in glob.glob(str(cell_path / 'final_spikes*.pkl'))])

    # unique + existing
    seen = set()
    cands = []
    for c in cand:
        k = str(c).lower()
        if k in seen:
            continue
        seen.add(k)
        if c.is_file():
            cands.append(c)

    if len(cands) == 0:
        return None

    # If caller requested an explicit state pkl (e.g., ...r0.pkl), keep it strict.
    if preferred_name and any(tok in str(preferred_name).lower() for tok in ['r0', 'r1', 'm0', 'm1']):
        for c in cands:
            if c.name.lower() == str(preferred_name).lower():
                return c

    best = None
    best_n = -1
    for c in cands:
        try:
            with open(c, 'rb') as f:
                payload = pickle.load(f)
            sp = _extract_spikes_from_pkl_payload(payload, len(full_vol))
            sp, _ = _align_spikes_to_full_trace_if_needed(sp, payload, full_vol, pkl_path=c, vol_sr=vol_sr)
            n_in = int(np.sum((sp >= int(i0)) & (sp < int(i1))))
            if n_in > best_n:
                best_n = n_in
                best = c
        except Exception:
            continue

    if best is not None:
        try:
            print(f'[PKL-SELECT] chose {best.name} for window ({i0/vol_sr:.3f}-{i1/vol_sr:.3f}s), spikes_in_window={best_n}')
        except Exception:
            pass
    return best



def _gaussian_smooth_1d(x, sigma_samples):
    x = np.asarray(x, dtype=float).ravel()
    sigma_samples = float(sigma_samples)
    if x.size == 0 or (not np.isfinite(sigma_samples)) or sigma_samples <= 0:
        return x.copy()
    radius = max(1, int(np.ceil(3.0 * sigma_samples)))
    k = np.arange(-radius, radius + 1, dtype=float)
    w = np.exp(-0.5 * (k / sigma_samples) ** 2)
    w /= np.sum(w)
    return np.convolve(x, w, mode='same')


def _safe_axis_limits(y, pad_frac=0.08, fallback=(0.0, 1.0)):
    y = np.asarray(y, dtype=float).ravel()
    y = y[np.isfinite(y)]
    if y.size == 0:
        return [float(fallback[0]), float(fallback[1])]

    lo = float(np.min(y))
    hi = float(np.max(y))
    span = hi - lo

    if not np.isfinite(lo) or not np.isfinite(hi):
        return [float(fallback[0]), float(fallback[1])]

    if span <= 0:
        pad = max(1e-3, 0.05 * abs(hi) + 1e-3)
    else:
        pad = max(1e-9, float(pad_frac) * span)

    return [lo - pad, hi + pad]


def plot_10s_window_with_spikes_and_overlay(
    cell_path,
    cal_sr,
    start_s,
    duration_s=10.0,
    vol_sr=500.0,
    save_stem='cal_vol_10s_window_with_spikes',
    spike_pkl_name='spike_detection_redefine_new.pkl',
    cal_smooth_sigma_s=0.03,
):
    """
    Two-subplot figure for a chosen time window:
    1) Voltage + detected spikes
    2) Combined voltage + calcium (separate y-axes)

    Inputs:
    - cell_path: cell folder containing traces and spike pkl
    - cal_sr: calcium sampling rate (Hz)
    - start_s: window start time (s)
    - duration_s: window duration (default 10 s)
    - cal_smooth_sigma_s: Gaussian sigma for calcium smoothing in seconds
    """
    cell_path = Path(cell_path)
    if not cell_path.is_dir():
        raise FileNotFoundError(f'Folder not found: {cell_path}')

    cal_file = _find_existing_file(cell_path, ['calDF.csv', 'calTraceDF.csv'])
    vol_file = _find_existing_file(cell_path, ['volDF.csv', 'volTraceDF.csv'])

    if cal_file is None:
        raise FileNotFoundError(f'No calcium file found in {cell_path} (expected calDF.csv or calTraceDF.csv)')
    if vol_file is None:
        raise FileNotFoundError(f'No voltage file found in {cell_path} (expected volDF.csv or volTraceDF.csv)')

    cal = _read_trace_csv_1d(cal_file)
    vol = _read_trace_csv_1d(vol_file)

    cal_sr = float(cal_sr)
    vol_sr = float(vol_sr)
    start_s = float(start_s)
    duration_s = float(duration_s)
    cal_smooth_sigma_s = float(cal_smooth_sigma_s)
    end_s = start_s + duration_s
    cal_sigma_samples = max(0.0, cal_smooth_sigma_s * cal_sr)

    # Build voltage window first (needed for smart PKL selection on motor segmented files)
    i0 = max(0, int(round(start_s * vol_sr)))
    i1 = min(len(vol), int(round(end_s * vol_sr)))
    if i1 <= i0:
        raise ValueError('Selected window is outside voltage trace range.')

    # Load spikes from pkl (preferred), fallback to voltage-based detection
    pkl_path = _choose_best_pkl_for_window(
        cell_path=cell_path,
        preferred_name=spike_pkl_name,
        full_vol=vol,
        i0=i0,
        i1=i1,
        vol_sr=vol_sr,
    )
    if pkl_path is None:
        pkl_path = _find_spike_pkl(cell_path, preferred_name=spike_pkl_name)

    spike_idx = np.array([], dtype=int)
    pkl_payload = None
    if pkl_path is not None:
        try:
            with open(pkl_path, 'rb') as f:
                pkl_payload = pickle.load(f)
            spike_idx = _extract_spikes_from_pkl_payload(pkl_payload, len(vol))
            spike_idx, _spike_offset_frames = _align_spikes_to_full_trace_if_needed(
                spike_idx,
                payload=pkl_payload,
                full_vol=vol,
                pkl_path=pkl_path,
                vol_sr=vol_sr,
            )
        except Exception:
            spike_idx = np.array([], dtype=int)
    if spike_idx.size == 0:
        spike_idx = _fallback_detect_spikes_from_voltage(vol, min_dist_frames=2)

    ci0 = max(0, int(round(start_s * cal_sr)))
    ci1 = min(len(cal), int(round(end_s * cal_sr)))
    if ci1 <= ci0:
        raise ValueError('Selected window is outside calcium trace range.')

    vol_w = vol[i0:i1]
    cal_w = cal[ci0:ci1]
    cal_w_smooth = _gaussian_smooth_1d(cal_w, cal_sigma_samples)
    t_vol = np.arange(i0, i1, dtype=float) / vol_sr
    t_cal = np.arange(ci0, ci1, dtype=float) / cal_sr

    spk_w = spike_idx[(spike_idx >= i0) & (spike_idx < i1)]

    # Robust finite-only limits (avoid capping from non-finite values)
    v_rng = _safe_axis_limits(vol_w, pad_frac=0.10)
    c_rng = _safe_axis_limits(cal_w_smooth, pad_frac=0.10)

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.5, 0.5],
        specs=[[{}], [{'secondary_y': True}]],
    )

    # 1) Voltage + spikes
    fig.add_trace(
        go.Scatter(
            x=t_vol,
            y=vol_w,
            mode='lines',
            line=dict(color='#d62728', width=1.2),
            name='Voltage',
            showlegend=False,
        ),
        row=1,
        col=1,
    )
    if spk_w.size > 0:
        y_spk = vol[spk_w]
        fig.add_trace(
            go.Scatter(
                x=spk_w.astype(float) / vol_sr,
                y=y_spk,
                mode='markers',
                marker=dict(color='black', size=7, symbol='circle'),
                name='Detected spikes',
                showlegend=False,
            ),
            row=1,
            col=1,
        )

    # 2) Combined overlay (dual y)
    fig.add_trace(
        go.Scatter(
            x=t_vol,
            y=vol_w,
            mode='lines',
            line=dict(color='#d62728', width=1.2),
            name='Voltage',
            showlegend=False,
        ),
        row=2,
        col=1,
        secondary_y=False,
    )
    if spk_w.size > 0:
        fig.add_trace(
            go.Scatter(
                x=spk_w.astype(float) / vol_sr,
                y=y_spk,
                mode='markers',
                marker=dict(color='black', size=6, symbol='circle'),
                name='Detected spikes',
                showlegend=False,
            ),
            row=2,
            col=1,
            secondary_y=False,
        )
    fig.add_trace(
        go.Scatter(
            x=t_cal,
            y=cal_w_smooth,
            mode='lines',
            line=dict(color='mediumseagreen', width=1.4),
            name='Calcium',
            showlegend=False,
        ),
        row=2,
        col=1,
        secondary_y=True,
    )

    # Bold Helvetica axis style
    common_axis_style = dict(
        showgrid=False,
        showline=True,
        linewidth=2.6,
        linecolor='black',
        mirror=False,
        ticks='outside',
        tickwidth=2.4,
        ticklen=7,
        tickfont=dict(size=14, family='Helvetica', color='black'),
    )
    fig.update_xaxes(**common_axis_style)
    fig.update_yaxes(**common_axis_style)

    fig.update_xaxes(
        title_text='<b>Time (s)</b>',
        title_font=dict(size=18, family='Helvetica', color='black'),
        range=[start_s, end_s],
        row=2,
        col=1,
    )
    fig.update_yaxes(
        title_text='<b>Voltage</b>',
        title_font=dict(size=18, family='Helvetica', color='black'),
        range=v_rng,
        row=1,
        col=1,
    )
    fig.update_yaxes(
        title_text='<b>Voltage (overlay)</b>',
        title_font=dict(size=16, family='Helvetica', color='black'),
        range=v_rng,
        row=2,
        col=1,
        secondary_y=False,
    )
    fig.update_yaxes(
        title_text='<b>Calcium dF/F</b>',
        title_font=dict(size=16, family='Helvetica', color='black'),
        range=c_rng,
        row=2,
        col=1,
        secondary_y=True,
    )

    fig.update_layout(
        template='simple_white',
        width=1500,
        height=680,
        font=dict(family='Helvetica', size=14, color='black'),
        showlegend=False,
        margin=dict(l=95, r=80, t=25, b=75),
    )

    html_path = cell_path / f'{save_stem}.html'
    svg_path = cell_path / f'{save_stem}.svg'

    fig.write_html(str(html_path))
    try:
        fig.write_image(str(svg_path))
    except Exception as e:
        print(f'[WARN] SVG export failed: {e}')

    pkl_name = pkl_path.name if pkl_path is not None else 'fallback_voltage_detection'
    print('Spike source:', pkl_name)
    print('Spikes in window:', int(spk_w.size))
    print('Saved HTML:', html_path)
    print('Saved SVG :', svg_path)
    return fig, html_path, svg_path

# ===== Example usage =====
RUN_FOR_SST = False  # True -> use SST-style final_spikes.pkl
CELL_PATH = r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\21-10-2025-MOTOR\fov7\cell0'  # <-- change
CAL_SR = 29.97  # <-- change
START_S =45.5  # <-- choose start time (s)
SPIKE_PKL_NAME = 'final_spikes.pkl' if RUN_FOR_SST else 'spike_detection_redefine_new.pkl'

fig, html_path, svg_path = plot_10s_window_with_spikes_and_overlay(
    cell_path=CELL_PATH,
    cal_sr=CAL_SR,
    start_s=START_S,
    duration_s=9,
    cal_smooth_sigma_s=0.06,
    vol_sr=500.0,
    save_stem='cal_vol_10s_window_with_spikes',
    spike_pkl_name=SPIKE_PKL_NAME,
)


[ALIGN] spike_detection_refined_newr0.pkl: shifted spikes by +12207 frames (24.414s) to full-trace timeline
[ALIGN] spike_detection_refined_newr1.pkl: shifted spikes by +43211 frames (86.422s) to full-trace timeline
[ALIGN] spike_detection_refined_newm1.pkl: shifted spikes by +27210 frames (54.420s) to full-trace timeline
[PKL-SELECT] chose spike_detection_refined_newr0.pkl for window (45.500-54.500s), spikes_in_window=282
[ALIGN] spike_detection_refined_newr0.pkl: shifted spikes by +12207 frames (24.414s) to full-trace timeline
Spike source: spike_detection_refined_newr0.pkl
Spikes in window: 282
Saved HTML: Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\21-10-2025-MOTOR\fov7\cell0\cal_vol_10s_window_with_spikes.html
Saved SVG : Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\21-10-2025-MOTOR\fov7\cell0\cal_vol_10s_window_with_spikes.svg


In [ ]:
### 10 s chosen-range figure (2 subplots)
# 1) Voltage + detected spikes
# 2) Combined voltage + calcium (separate y-axes)

RUN_FOR_SST = True  # True -> use SST-style final_spikes.pkl
CELL_PATH = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc46\R\07-01-2025-ans\cell0"  # <-- change
CAL_SR = 30.0            # <-- change
START_S = 10.0           # <-- choose start time (seconds)
WINDOW_S = 10.0          # keep 10 s
SPIKE_PKL_NAME = 'final_spikes.pkl' if RUN_FOR_SST else 'spike_detection_redefine_new.pkl'

fig, html_path, svg_path = plot_10s_window_with_spikes_and_overlay(
    cell_path=CELL_PATH,
    cal_sr=CAL_SR,
    start_s=START_S,
    duration_s=WINDOW_S,
    vol_sr=500.0,
    save_stem=f"cal_vol_10s_window_{START_S:.2f}s",
    spike_pkl_name=SPIKE_PKL_NAME,
)


NameError: name 'plot_10s_window_with_spikes_and_overlay' is not defined

In [1]:
# MULTI_CELL_SVG_STACK_V1
# Build one stacked SVG from multiple cell folders.
# Figure width: 18.5 cm
# Row height per trace: 1.4 cm
# Each row plots Voltage + Calcium with separate y-axes.
# X axis is hidden on all rows; only last row gets a 2 s scale bar.

from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms


def _read_trace_csv_1d(csv_path):
    arr = pd.read_csv(csv_path).to_numpy(dtype=float).ravel()
    arr = np.asarray(arr, dtype=float)
    return arr[np.isfinite(arr)]


def _find_existing_file(base_path, candidates):
    base_path = Path(base_path)
    for name in candidates:
        p = base_path / name
        if p.is_file():
            return p
    return None


def _gaussian_smooth_1d(x, sigma_samples):
    x = np.asarray(x, dtype=float).ravel()
    sigma_samples = float(sigma_samples)
    if x.size == 0 or (not np.isfinite(sigma_samples)) or sigma_samples <= 0:
        return x.copy()
    radius = max(1, int(np.ceil(3.0 * sigma_samples)))
    k = np.arange(-radius, radius + 1, dtype=float)
    w = np.exp(-0.5 * (k / sigma_samples) ** 2)
    w /= np.sum(w)
    return np.convolve(x, w, mode='same')


def _cm_to_in(cm):
    return float(cm) / 2.54


def _norm_folder_key(path_like):
    s = str(path_like).strip().strip('"').strip("'")
    s = s.rstrip('\/').strip()
    return os.path.normcase(os.path.normpath(s))


def _build_cal_sr_lookup_from_pyrlow(pyrlow_csv):
    p = Path(pyrlow_csv)
    if not p.is_file():
        raise FileNotFoundError(f'PyrLow CSV not found: {p}')

    df = pd.read_csv(p)
    need = {'Link', 'CALsr'}
    if not need.issubset(set(df.columns)):
        raise ValueError(f'PyrLow CSV must include columns {need}; got {list(df.columns)}')

    lookup = {}
    for _, row in df.iterrows():
        link = row.get('Link', None)
        calsr = row.get('CALsr', np.nan)
        if not isinstance(link, str) or len(link.strip()) == 0:
            continue
        try:
            calsr = float(calsr)
        except Exception:
            continue
        if not np.isfinite(calsr) or calsr <= 0:
            continue
        lookup[_norm_folder_key(link)] = calsr
    return lookup


def _resolve_cal_sr_for_cell(cell_folder, cal_sr_lookup, default_cal_sr=30.0, strict=True):
    key = _norm_folder_key(cell_folder)
    if key in cal_sr_lookup:
        return float(cal_sr_lookup[key])
    if strict:
        raise KeyError(f'No CALsr found in PyrLow table for cell folder: {cell_folder}')
    return float(default_cal_sr)


def _load_windowed_cell_traces(cell_folder, cal_sr, vol_sr, start_s=0.0, duration_s=None, cal_smooth_sigma_s=0.0):
    cell_folder = Path(cell_folder)
    cal_file = _find_existing_file(cell_folder, ['calDF.csv', 'calTraceDF.csv'])
    vol_file = _find_existing_file(cell_folder, ['volDF.csv', 'volTraceDF.csv'])
    if cal_file is None:
        raise FileNotFoundError(f'No calcium file found in {cell_folder}')
    if vol_file is None:
        raise FileNotFoundError(f'No voltage file found in {cell_folder}')

    cal = _read_trace_csv_1d(cal_file)
    vol = _read_trace_csv_1d(vol_file)

    max_t = min(len(cal) / float(cal_sr), len(vol) / float(vol_sr))
    t0 = max(0.0, float(start_s))
    t1 = max_t if duration_s is None else min(max_t, t0 + float(duration_s))
    if t1 <= t0:
        raise ValueError(f'Invalid time window for {cell_folder}: start={t0}, end={t1}')

    ci0 = int(round(t0 * cal_sr))
    ci1 = int(round(t1 * cal_sr))
    vi0 = int(round(t0 * vol_sr))
    vi1 = int(round(t1 * vol_sr))

    cal_w = cal[ci0:ci1]
    vol_w = vol[vi0:vi1]

    cal_sigma_samples = max(0.0, float(cal_smooth_sigma_s) * float(cal_sr))
    cal_w = _gaussian_smooth_1d(cal_w, cal_sigma_samples)

    t_cal = np.arange(ci0, ci1, dtype=float) / float(cal_sr) - t0
    t_vol = np.arange(vi0, vi1, dtype=float) / float(vol_sr) - t0

    return {
        'folder': str(cell_folder),
        'name': cell_folder.name,
        't_cal': t_cal,
        'cal': cal_w,
        't_vol': t_vol,
        'vol': vol_w,
        'window_s': float(t1 - t0),
    }


def plot_multi_cell_cal_vol_svg(
    cell_folders,
    out_svg,
    cal_sr=30.0,
    vol_sr=500.0,
    start_s=0.0,
    duration_s=None,
    fig_width_cm=18.5,
    row_height_cm=1.4,
    cal_smooth_sigma_s=0.0,
    scale_bar_s=2.0,
    vol_color='#ff00ff',
    cal_color='#2ca02c',
    hide_y_ticks=True,
    pyrlow_csv=r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR.csv',
    strict_cal_sr_lookup=False,
):
    records = []
    skipped = []
    cal_sr_lookup = _build_cal_sr_lookup_from_pyrlow(pyrlow_csv)
    for folder in cell_folders:
        try:
            folder_clean = str(folder).strip()
            cal_sr_cell = _resolve_cal_sr_for_cell(
                folder_clean,
                cal_sr_lookup=cal_sr_lookup,
                default_cal_sr=float(locals().get('cal_sr', 30.0)),
                strict=bool(strict_cal_sr_lookup),
            )
            rec = _load_windowed_cell_traces(
                folder_clean,
                cal_sr=float(cal_sr_cell),
                vol_sr=float(vol_sr),
                start_s=float(start_s),
                duration_s=duration_s,
                cal_smooth_sigma_s=float(cal_smooth_sigma_s),
            )
            rec['cal_sr'] = float(cal_sr_cell)
            records.append(rec)
        except Exception as e:
            msg = f'[WARN] skipped {folder}: {e}'
            print(msg)
            skipped.append(msg)

    if len(records) == 0:
        details = '\n'.join(skipped[:10]) if len(skipped) else 'no folder diagnostics captured'
        raise RuntimeError('No valid folders to plot. Details:\n' + details)

    n = len(records)
    fig_w_in = _cm_to_in(fig_width_cm)
    fig_h_in = _cm_to_in(row_height_cm * n)

    fig, axes = plt.subplots(
        nrows=n,
        ncols=1,
        figsize=(fig_w_in, fig_h_in),
        sharex=True,
        squeeze=False,
    )

    axes = axes.ravel()
    twin_axes = []

    x_max = max(float(rec['window_s']) for rec in records)

    for i, (ax, rec) in enumerate(zip(axes, records)):
        ax2 = ax.twinx()
        twin_axes.append(ax2)

        ax.plot(rec['t_vol'], rec['vol'], color=vol_color, lw=0.7, alpha=0.9)
        ax2.plot(rec['t_cal'], rec['cal'], color=cal_color, lw=0.7, alpha=0.9)

        ax.set_xlim(0.0, x_max)

        # Hide x axis on all rows (transparent/clean look)
        ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
        ax.spines['bottom'].set_visible(False)

        # Keep only left and right y spines for voltage/calcium separation
        ax.spines['top'].set_visible(False)
        ax2.spines['top'].set_visible(False)

        if hide_y_ticks:
            ax.tick_params(axis='y', left=False, labelleft=False)
            ax2.tick_params(axis='y', right=False, labelright=False)

        # No per-cell/CALsr label
        ax.text(
            0.002, 0.88, "",
            transform=ax.transAxes,
            ha='left', va='top', fontsize=7, color='black'
        )

    # Add 2 s scale bar only under the last subplot
    last_ax = axes[-1]
    span = x_max
    bar_len = float(scale_bar_s)
    if span <= 0:
        bar_len = 2.0
        span = 2.5
        last_ax.set_xlim(0.0, span)

    x1 = span * 0.98
    x0 = max(0.0, x1 - bar_len)
    trans = mtransforms.blended_transform_factory(last_ax.transData, last_ax.transAxes)

    last_ax.plot([x0, x1], [-0.18, -0.18], transform=trans, color='black', lw=1.4, clip_on=False)
    last_ax.text((x0 + x1) / 2.0, -0.24, f'{bar_len:g} s', transform=trans,
                 ha='center', va='top', fontsize=8, color='black')

    fig.subplots_adjust(left=0.06, right=0.94, top=0.99, bottom=0.08, hspace=0.08)

    out_svg = Path(out_svg)
    out_svg.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_svg, format='svg', transparent=True)
    plt.close(fig)
    print('Saved SVG:', out_svg)
    return out_svg


# ------------------ Usage ------------------
CELL_FOLDERS = [#r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\19-11-2025-awake\fov19\cell1',
#                 r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\RL-REAL\fov2\cell1',
                r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc46\R\07-01-2025-ans\cell0',
                r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc46\R\07-01-2025-ans\cell3',
                #r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\RL-REAL\FOV1\cell4',
                r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC44\L\26-11-2025-ANST\fov7\cell1',
                #r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC44\L\12-01-2025-awake\fov18\cell1',
                #r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\RUGC44\L\12-01-2025-awake\fov18\cell2',
                r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\rugc42\Wh\21-10-2025-MOTOR\fov7\cell0'

    # r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\...\cell0",
    # r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\...\cell1",
]

OUT_SVG = r"E:\presentation\final_figure\Pyr\forFigure_multi_cells.svg"

# Set duration_s=None to use full overlap duration in each folder.
# Use start_s/duration_s to align a specific window across all folders.
plot_multi_cell_cal_vol_svg(
    cell_folders=CELL_FOLDERS,
    out_svg=OUT_SVG,
    vol_sr=500.0,
    start_s=0.0,
    duration_s=60.0,
    fig_width_cm=18.5,
    row_height_cm=1.4,
    cal_smooth_sigma_s=0.06,
    scale_bar_s=2.0,
    vol_color='#ff00ff',
    pyrlow_csv=r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR.csv',
    strict_cal_sr_lookup=False,
)



Saved SVG: E:\presentation\final_figure\Pyr\forFigure_multi_cells.svg


WindowsPath('E:/presentation/final_figure/Pyr/forFigure_multi_cells.svg')